# Required libraries

In [1]:
import scanpy as sc
import numpy as np
import os
import random
import warnings
from sklearn.cluster import (
    KMeans,
    SpectralClustering,
    AffinityPropagation,
    MeanShift,
    DBSCAN,
    OPTICS,
    Birch,
    estimate_bandwidth,
)
from scanpy.tl import leiden
from sklearn.metrics import pairwise_distances
from validclust import cop
import rpy2.robjects as ro
from rpy2.robjects import numpy2ri
from rpy2.robjects.packages import importr
import docx
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
clcrit = importr("clusterCrit")
warnings.filterwarnings("ignore")
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Preprocessing

In [2]:
files = {"PBMC3K": "pbmc3k_raw.h5ad",
         "Liver_Biliary": "Ma2019_Liver-Biliary.h5ad",
         "Ovarian_2021": "Olalekan2021_Ovarian.h5ad",
         "Ovarian_2018": "Shih2018_Ovarian.h5ad"
        }
adatas = {}
for name, path in files.items():
    adata = sc.read_h5ad(path)
    adata.var_names_make_unique()
    adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")
    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=["mt", "ribo", "hb"],
        inplace=True
    )
    adata = adata[adata.obs.pct_counts_mt < 5].copy()
    sc.pp.filter_cells(adata, min_genes=100)
    sc.pp.filter_genes(adata, min_cells=3)
    adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=350, subset=True)
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, random_state=SEED)
    sc.pp.neighbors(adata, random_state=SEED)
    sc.tl.umap(adata, random_state=SEED)
    X = adata.obsm["X_pca"]
    adata.obsm["X_tsne"] = TSNE(
        n_components=2,
        random_state=SEED,
        perplexity=30
    ).fit_transform(X)
    adatas[name] = adata

# Tetrad Triangular Function (TTF)

In [3]:
def wg_index(X, labels):
    l_vec = ro.IntVector(labels)
    x_vec = numpy2ri.py2rpy(X)
    a = clcrit.intCriteria(x_vec, l_vec, "wemmert_gancarski")
    return list(a[0])[0]
def csl_index(X, labels):
    unique_labels = np.unique(labels)
    centroids = np.array([X[labels == k].mean(axis=0) for k in unique_labels])
    numerator = 0.0
    for i, k in enumerate(unique_labels):
        cluster_points = X[labels == k]
        n_k = len(cluster_points)
        if n_k <= 1:
            continue
        intra_dists = pairwise_distances(cluster_points)
        numerator += np.max(intra_dists) / n_k
    inter_dists = pairwise_distances(centroids)
    denominator = 0.0
    for i in range(len(unique_labels)):
        mask = np.ones(len(unique_labels), dtype=bool)
        mask[i] = False
        denominator += np.min(inter_dists[i][mask])
    if denominator == 0:
        return np.nan
    return numerator / denominator
def cop_index(X, labels):
    dist = pairwise_distances(X)
    return cop(X, dist, labels)
def rs_index(X, labels):
    unique_labels = np.unique(labels)
    global_centroid = np.mean(X, axis=0)
    tss = np.sum((X - global_centroid) ** 2)
    wss = 0.0
    for k in unique_labels:
        cluster_points = X[labels == k]
        if len(cluster_points) > 0:
            centroid_k = cluster_points.mean(axis=0)
            wss += np.sum((cluster_points - centroid_k) ** 2)
    bss = tss - wss
    if tss == 0:
        return np.nan
    return bss / tss
def norm(x):
    return 1 / (1 + np.exp(-x))
def TT(adata, labels, W):
    # W -> weight vector
    X = adata.obsm["X_pca"]
    am = bm = cm = dm = np.finfo(float).eps
    try:
        am = rs_index(X, labels)
    except:
        pass
    try:
        bm = norm(wg_index(X, labels))
    except:
        pass
    try:
        cm = 1 - norm(csl_index(X, labels))
    except:
        pass
    try:
        dm = 1 - cop_index(X, labels)
    except:
        pass
    return (am * W[0] + cm * W[2]) * (bm * W[1] + dm * W[3]) / 2

# Clustering algorithms

In [4]:
def leiden_alg(adata):
    sc.tl.leiden(adata)
    labels = adata.obs["leiden"].astype(int).values
    return labels
def optics_alg(adata):
    X = adata.obsm["X_pca"]
    optics = OPTICS(min_samples=10, xi=0.05, min_cluster_size=0.05)
    adata.obs["OPTICS"] = optics.fit_predict(X).astype(str)
    labels = adata.obs["OPTICS"].astype(int).values
    return labels
def birch_alg(adata):
    X = adata.obsm["X_pca"]
    birch = Birch(n_clusters=10)
    adata.obs["BIRCH"] = birch.fit_predict(X).astype(str)
    labels = adata.obs["BIRCH"].astype(int).values
    return labels
def kmeans_alg(adata):
    X = adata.obsm["X_pca"]
    kmeans = KMeans(n_clusters=8, n_init=10)
    labels = kmeans.fit(X).labels_
    return labels
def spectral_alg(adata):
    X = adata.obsm["X_pca"]
    spectral = SpectralClustering(
        n_clusters=8,
        affinity="nearest_neighbors",)
    labels = spectral.fit(X).labels_
    return labels
def affinity_alg(adata):
    X = adata.obsm["X_pca"]
    ap = AffinityPropagation()
    labels = ap.fit_predict(X)
    return labels
def ms_alg(adata):
    X = adata.obsm["X_pca"]
    bandwidth = estimate_bandwidth(X, quantile=0.2, n_samples=500)
    ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)
    labels = ms.fit_predict(X)
    return labels
def dbscan_alg(adata):
    X = adata.obsm["X_pca"]
    dbscan = DBSCAN(eps=10, min_samples=10)
    labels = dbscan.fit_predict(X)
    return labels

# Docx maker

In [5]:
def get_df(algs, rewards):
    d = {}
    for i, alg in enumerate(algs):
        d[alg.__name__.replace("_alg", "")] = [rewards[i]]
    d = {k: v for k, v in sorted(d.items(), key=lambda item: item[1][0])}
    return pd.DataFrame(d)
def save_to_docx(algs, rewards, run, data_name):
    df = get_df(algs, rewards)
    file_name = f"{data_name}.docx"
    doc = None
    if os.path.isfile(file_name):
        doc = docx.Document(file_name)
    else:
        doc = docx.Document()
    doc.add_paragraph(f"\n\n\n run {run}:")
    t = doc.add_table(df.shape[0] + 1, df.shape[1])
    for j in range(df.shape[-1]):
        t.cell(0, j).text = df.columns[j]
    for i in range(df.shape[0]):
        for j in range(df.shape[-1]):
            t.cell(i + 1, j).text = str(df.values[i, j])
    doc.save(file_name)

# Plot Rewards

In [6]:
def plot_rewards(alg_name, reward_history, max_iter):
    plt.plot(range(max_iter), reward_history)
    plt.xticks(range(0, max_iter + 1, 10))
    plt.xlabel("Iteration")
    plt.ylabel("Reward")
    plt.title(f"Reward history of {alg_name}")
    plt.show()

# RLCS

In [7]:
def RLCS(cluster_algs, w, adata, data_name):
    N = len(cluster_algs)
    MaxIteration = 50
    CReward = [0] * N
    Run = [0] * N
    TTF_0 = 0
    reward_history = dict([(i.__name__.replace("_alg", ""), list()) for i in cluster_algs])
    save_at_step = 5
    save_at_list = list(range(save_at_step, MaxIteration, save_at_step))
    save_at_list.append(MaxIteration - 1)
    print("First run:")
    for i, alg in enumerate(cluster_algs):
        print(alg.__name__.replace("_alg", ""), end="\t")
        labels = alg(adata)
        TTF = TT(adata, labels, w)
        print("TTF:", TTF)
        CReward[i] = TTF
        TTF_0 = TTF
    RLalpha = 0
    print()
    print("Main loop:")
    for t in range(MaxIteration):
        chN = np.random.random()
        if chN <= RLalpha:
            i = CReward.index(max(CReward))
        else:
            i = np.random.randint(0, N)
        print(
            f"{t+1}/{MaxIteration}", cluster_algs[i].__name__.replace("_alg", ""), end="\t"
        )
        labels = cluster_algs[i](adata)
        TTF = TT(adata, labels, w)
        print("TTF:", TTF)
        CReward[i] += 1
        if TTF == TTF_0:
            CReward[i] = max(0, CReward[i] - ((sum(CReward) - CReward[i]) / N))
        RLalpha += 1 / MaxIteration
        TTF_0 = TTF
        Run[i] += 1
        for n, alg in enumerate(cluster_algs):
            reward_history[alg.__name__.replace("_alg", "")].append(CReward[n])
    save_to_docx(cluster_algs, CReward, t, data_name) 

# Execution 

In [8]:
cluster_algs = [
    leiden_alg,
    optics_alg,
    birch_alg,
    kmeans_alg,
    spectral_alg,
    affinity_alg,
    ms_alg,
    dbscan_alg,
]
w = [1, 1, 1, 1]
for name, adata in adatas.items():
    print(f"Running RLCS on {name}")
    RLCS(cluster_algs, w, adata, name)

Running RLCS on PBMC3K
First run:
leiden	TTF: 0.3582631775381255
optics	TTF: 8.116380663447721e-17
birch	TTF: 0.34971014912514053
kmeans	TTF: 0.38631284368845953
spectral	TTF: 0.3636032360288687
affinity	TTF: 0.4978093100423552
ms	TTF: 0.3784228639176958
dbscan	TTF: 0.3413519340500181

Main loop:
1/50 leiden	TTF: 0.3582631775381255
2/50 dbscan	TTF: 0.3413519340500181
3/50 spectral	TTF: 0.3636367918307433
4/50 affinity	TTF: 0.4978093100423552
5/50 leiden	TTF: 0.3582631775381255
6/50 leiden	TTF: 0.3582631775381255
7/50 kmeans	TTF: 0.382060088828021
8/50 kmeans	TTF: 0.383019822463891
9/50 optics	TTF: 8.116380663447721e-17
10/50 leiden	TTF: 0.3582631775381255
11/50 spectral	TTF: 0.36377265790403474
12/50 spectral	TTF: 0.3636654452721205
13/50 birch	TTF: 0.34971014912514053
14/50 kmeans	TTF: 0.3858647528883636
15/50 leiden	TTF: 0.3582631775381255
16/50 kmeans	TTF: 0.3842968083756445
17/50 leiden	TTF: 0.3582631775381255
18/50 ms	TTF: 0.3784228639176958
19/50 leiden	TTF: 0.3582631775381255
20